In [ ]:
import warnings, glob
import numpy as np
import pandas as pd
from google.colab import drive
from sklearn.feature_selection import mutual_info_classif, f_classif, RFE
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score, f1_score, recall_score

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

warnings.filterwarnings("ignore")
drive.mount("/content/drive")
print("XGBoost available:", HAS_XGB)

Mounted at /content/drive
XGBoost available: True


In [ ]:
FOLDER = "/content/drive/MyDrive/Thesis/radar/JamilIsp-RADAR-cb0c4c2/dataset-imbalanced-10"
P = sorted(glob.glob(f"{FOLDER}/**/*.csv", recursive=True))[0]
print("using:", P)

df = pd.read_csv(P, usecols=["@timestamp", "event.code", "target-class-name", "target-class"],
                 low_memory=False)

# Replace "24:" hours representation if occurring at start of string to make them parsable,
# or parse with errors='coerce' to turn them to NaT and drop them.
df["@timestamp"] = pd.to_datetime(df["@timestamp"], errors='coerce')
df = df.dropna(subset=["@timestamp"])

df["bin"] = df["@timestamp"].dt.floor("10s")
t0 = df["bin"].min()
block = ((df["bin"] - t0) // pd.Timedelta("300s")).astype(int)
df["grp"] = np.where(df["target-class-name"] == "goodware",
                     "goodware_block" + block.astype(str),
                     df["target-class-name"])
df["unit"] = df["grp"] + "|" + df["bin"].astype(str)
print(df.shape, "| units:", df["unit"].nunique(), "| groups:", df["grp"].nunique())

using: /content/drive/MyDrive/Thesis/radar/JamilIsp-RADAR-cb0c4c2/dataset-imbalanced-10/dataset-imbalanced-10-all.csv
(117443, 7) | units: 9865 | groups: 302


In [ ]:
X = pd.crosstab(df["unit"], df["event.code"])
X.columns = [f"ev_{int(c)}" for c in X.columns]
X = X.drop(columns=[c for c in ["ev_255"] if c in X.columns])
meta = df.groupby("unit").agg(grp=("grp", "first"), y=("target-class", "first")).loc[X.index]
X = X[X.columns[X.nunique() > 1]]

cols = list(X.columns)
Xv = X.values.astype(float)
yv = meta["y"].values.astype(int)
gv = meta["grp"].values
print("feature table:", X.shape, "| labels (0,1):", np.bincount(yv))

feature table: (9865, 18) | labels (0,1): [6761 3104]


In [ ]:
def paper1_votes(Xs, y):
    n = Xs.shape[1]
    mi = mutual_info_classif(Xs, y, random_state=42)
    F, _ = f_classif(Xs, y)
    F = np.nan_to_num(F)
    k = min(10, n - 1)
    rfe = RFE(LogisticRegression(max_iter=5000), n_features_to_select=k, step=1).fit(Xs, y)
    corr = np.nan_to_num(np.array([abs(np.corrcoef(Xs[:, j], y)[0, 1]) for j in range(n)]))

    votes = np.zeros(n, dtype=int)
    votes += (mi > mi.mean())          # Method 1: MI above mean
    votes += (F > F.mean())            # Method 2: F above mean
    votes += (rfe.ranking_ == 1)       # Method 3: RFE kept (rank 1)
    votes += (corr > corr.mean())      # Method 4: |corr| above mean

    selected = np.where(votes / 4 >= 0.5)[0]   # V_i >= 0.5
    if len(selected) == 0:
        selected = np.arange(n)
    return votes, selected, mi

def order_from(Xs, y):
    votes, _, mi = paper1_votes(Xs, y)
    return np.lexsort((-mi, -votes))   # most votes first, tie-break by MI

In [ ]:
def make_models():
    m = {
        "LR": make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
        "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
        "NB": GaussianNB(),
        "DT": DecisionTreeClassifier(random_state=42),
        "RF": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
        "AB": AdaBoostClassifier(n_estimators=50, random_state=42),
    }
    if HAS_XGB:
        m["XGB"] = XGBClassifier(n_estimators=50, max_depth=3, learning_rate=0.1,
                                 random_state=42, eval_metric="logloss")
    return m

def fit_eval(Xtr, ytr, Xte, yte, idx, name):
    model = make_models()[name]
    model.fit(Xtr[:, idx], ytr)
    pred = model.predict(Xte[:, idx])
    return (balanced_accuracy_score(yte, pred),
            f1_score(yte, pred, zero_division=0),
            1 - recall_score(yte, pred, zero_division=0))

In [ ]:
gkf = GroupKFold(n_splits=5)
folds = list(gkf.split(Xv, yv, gv))
for i, (tr, te) in enumerate(folds):
    print(f"fold {i}: train {len(tr)} (ransom {yv[tr].sum()}), "
          f"test {len(te)} (ransom {yv[te].sum()})")

fold 0: train 7890 (ransom 2487), test 1975 (ransom 617)
fold 1: train 7890 (ransom 2489), test 1975 (ransom 615)
fold 2: train 7894 (ransom 2469), test 1971 (ransom 635)
fold 3: train 7891 (ransom 2480), test 1974 (ransom 624)
fold 4: train 7895 (ransom 2491), test 1970 (ransom 613)


In [ ]:
p1_rows = []
for i, (tr, te) in enumerate(folds):
    sc = StandardScaler().fit(Xv[tr])
    _, sel, _ = paper1_votes(sc.transform(Xv[tr]), yv[tr])
    for name in make_models():
        bal, f1, fnr = fit_eval(Xv[tr], yv[tr], Xv[te], yv[te], sel, name)
        p1_rows.append({"fold": i, "n_sel": len(sel), "model": name,
                        "bal_acc": bal, "F1": f1, "FNR": fnr})
p1 = pd.DataFrame(p1_rows)
print(p1.groupby("model")[["bal_acc", "F1", "FNR"]].agg(["mean", "std"]).round(3))
print("selected per fold:", p1.drop_duplicates("fold")["n_sel"].tolist())

      bal_acc            F1           FNR       
         mean    std   mean    std   mean    std
model                                           
AB      0.561  0.025  0.250  0.079  0.843  0.059
DT      0.593  0.044  0.399  0.079  0.670  0.082
KNN     0.699  0.018  0.601  0.020  0.097  0.037
LR      0.516  0.017  0.073  0.067  0.960  0.037
NB      0.618  0.020  0.542  0.018  0.054  0.038
RF      0.597  0.042  0.405  0.076  0.667  0.079
XGB     0.581  0.038  0.323  0.116  0.773  0.093
selected per fold: [10, 9, 9, 10, 9]


In [ ]:
PCTS = list(range(10, 101, 10))

def n_for(p, total):
    return max(1, int(round(p / 100 * total)))

sweep_rows = []
for i, (tr, te) in enumerate(folds):
    sc = StandardScaler().fit(Xv[tr])
    order = order_from(sc.transform(Xv[tr]), yv[tr])
    for p in PCTS:
        idx = order[:n_for(p, len(order))]
        for name in make_models():
            bal, f1, fnr = fit_eval(Xv[tr], yv[tr], Xv[te], yv[te], idx, name)
            sweep_rows.append({"fold": i, "percent": p, "model": name,
                               "bal_acc": bal, "F1": f1, "FNR": fnr})
sweep = pd.DataFrame(sweep_rows)
print(sweep.groupby(["model", "percent"])["bal_acc"].agg(["mean", "std"])
            .round(3).unstack("model"))

          mean                                              std                \
model       AB     DT    KNN     LR     NB     RF    XGB     AB     DT    KNN   
percent                                                                         
10       0.500  0.501  0.581  0.500  0.577  0.501  0.500  0.000  0.003  0.020   
20       0.516  0.584  0.626  0.500  0.592  0.584  0.581  0.030  0.046  0.033   
30       0.516  0.583  0.646  0.501  0.600  0.584  0.583  0.030  0.047  0.023   
40       0.561  0.581  0.669  0.507  0.607  0.584  0.561  0.025  0.044  0.024   
50       0.561  0.592  0.699  0.513  0.616  0.596  0.581  0.025  0.043  0.021   
60       0.556  0.613  0.706  0.521  0.617  0.620  0.582  0.023  0.030  0.026   
70       0.556  0.614  0.706  0.521  0.614  0.620  0.582  0.023  0.031  0.024   
80       0.556  0.615  0.706  0.521  0.615  0.625  0.583  0.023  0.030  0.024   
90       0.556  0.627  0.714  0.523  0.614  0.637  0.582  0.023  0.033  0.022   
100      0.556  0.627  0.716

In [ ]:
def choose_percent(X_train, y_train, g_train, target_model_name="RF"):
    """
    Performs internal GroupKFold cross-validation on the training set to
    select the optimal feature percentage based on mean balanced accuracy.
    """
    inner_gkf = GroupKFold(n_splits=5)
    inner_folds = list(inner_gkf.split(X_train, y_train, g_train))

    perf_pcts = {p: [] for p in PCTS}

    for itr, ite in inner_folds:
        sc = StandardScaler().fit(X_train[itr])
        order = order_from(sc.transform(X_train[itr]), y_train[itr])

        for p in PCTS:
            idx = order[:n_for(p, len(order))]
            model = make_models()[target_model_name]
            model.fit(X_train[itr][:, idx], y_train[itr])
            pred = model.predict(X_train[ite][:, idx])
            perf_pcts[p].append(balanced_accuracy_score(y_train[ite], pred))

    # Choose the percent with the highest mean balanced accuracy
    best_p = max(PCTS, key=lambda p: np.mean(perf_pcts[p]))
    return best_p

final_rows = []
for i, (tr, te) in enumerate(folds):
    p_star = choose_percent(Xv[tr], yv[tr], gv[tr], "RF")
    sc = StandardScaler().fit(Xv[tr])
    order = order_from(sc.transform(Xv[tr]), yv[tr])
    idx = order[:n_for(p_star, len(order))]
    for name in make_models():
        bal, f1, fnr = fit_eval(Xv[tr], yv[tr], Xv[te], yv[te], idx, name)
        final_rows.append({"fold": i, "chosen_percent": p_star, "n_sel": len(idx),
                           "model": name, "bal_acc": bal, "F1": f1, "FNR": fnr})
final = pd.DataFrame(final_rows)
print(final.groupby("model")[["bal_acc", "F1", "FNR"]].agg(["mean", "std"]).round(3))
print("chosen percent per fold:", final.drop_duplicates("fold")["chosen_percent"].tolist())

      bal_acc            F1           FNR       
         mean    std   mean    std   mean    std
model                                           
AB      0.556  0.023  0.224  0.063  0.867  0.040
DT      0.621  0.029  0.457  0.050  0.601  0.065
KNN     0.713  0.028  0.613  0.028  0.125  0.043
LR      0.521  0.021  0.113  0.077  0.935  0.047
NB      0.613  0.014  0.539  0.013  0.046  0.033
RF      0.630  0.026  0.471  0.045  0.591  0.059
XGB     0.583  0.038  0.324  0.117  0.773  0.095
chosen percent per fold: [100, 100, 100, 60, 100]


In [ ]:
all_idx = np.arange(len(cols))
base_rows = []
for i, (tr, te) in enumerate(folds):
    for name in make_models():
        bal, f1, fnr = fit_eval(Xv[tr], yv[tr], Xv[te], yv[te], all_idx, name)
        base_rows.append({"fold": i, "model": name, "bal_acc": bal, "F1": f1, "FNR": fnr})
base = pd.DataFrame(base_rows)
print(base.groupby("model")[["bal_acc", "F1", "FNR"]].agg(["mean", "std"]).round(3))

      bal_acc            F1           FNR       
         mean    std   mean    std   mean    std
model                                           
AB      0.556  0.023  0.224  0.063  0.867  0.040
DT      0.627  0.034  0.468  0.060  0.581  0.080
KNN     0.715  0.025  0.615  0.026  0.132  0.049
LR      0.530  0.028  0.149  0.100  0.911  0.064
NB      0.614  0.013  0.540  0.011  0.040  0.024
RF      0.637  0.033  0.483  0.057  0.571  0.075
XGB     0.583  0.038  0.324  0.116  0.774  0.095


In [ ]:
summary = pd.DataFrame({
    "all_features": base.groupby("model")["bal_acc"].mean(),
    "paper1_rule": p1.groupby("model")["bal_acc"].mean(),
    "nested_sweep": final.groupby("model")["bal_acc"].mean(),
}).round(3)
print(summary)

       all_features  paper1_rule  nested_sweep
model                                         
AB            0.556        0.561         0.556
DT            0.627        0.593         0.621
KNN           0.715        0.699         0.713
LR            0.530        0.516         0.521
NB            0.614        0.618         0.613
RF            0.637        0.597         0.630
XGB           0.583        0.581         0.583


In [ ]:
sc_all = StandardScaler().fit(Xv)
votes_all, sel_all, mi_all = paper1_votes(sc_all.transform(Xv), yv)
order_all = np.lexsort((-mi_all, -votes_all))
sel_set = set(sel_all.tolist())
report = pd.DataFrame({
    "feature": [cols[j] for j in order_all],
    "votes": votes_all[order_all],
    "MI": mi_all[order_all].round(4),
    "in_paper_rule": [j in sel_set for j in order_all],
})
print(report.to_string(index=False))

sets = []
for tr, te in folds:
    sc = StandardScaler().fit(Xv[tr])
    _, sel, _ = paper1_votes(sc.transform(Xv[tr]), yv[tr])
    sets.append(set(sel.tolist()))
jac = [len(a & b) / len(a | b) for i, a in enumerate(sets) for b in sets[i + 1:]]
print("paper-rule selected-set Jaccard, mean:", round(float(np.mean(jac)), 3))

feature  votes     MI  in_paper_rule
   ev_2      4 0.0269           True
  ev_11      4 0.0243           True
   ev_7      4 0.0196           True
  ev_23      3 0.0270           True
  ev_10      3 0.0225           True
  ev_29      3 0.0118           True
  ev_22      3 0.0015           True
  ev_13      2 0.0202           True
   ev_5      2 0.0051           True
   ev_3      2 0.0000           True
   ev_6      1 0.0000          False
   ev_1      0 0.0087          False
   ev_8      0 0.0087          False
  ev_12      0 0.0055          False
  ev_17      0 0.0043          False
  ev_25      0 0.0021          False
   ev_4      0 0.0000          False
  ev_15      0 0.0000          False
paper-rule selected-set Jaccard, mean: 0.77


In [ ]:
print(final.round(3).to_string(index=False))

 fold  chosen_percent  n_sel model  bal_acc    F1   FNR
    0             100     18    LR    0.534 0.159 0.909
    0             100     18   KNN    0.734 0.631 0.088
    0             100     18    NB    0.624 0.545 0.023
    0             100     18    DT    0.627 0.466 0.596
    0             100     18    RF    0.642 0.490 0.574
    0             100     18    AB    0.569 0.259 0.846
    0             100     18   XGB    0.600 0.378 0.733
    1             100     18    LR    0.550 0.219 0.868
    1             100     18   KNN    0.685 0.581 0.193
    1             100     18    NB    0.622 0.540 0.065
    1             100     18    DT    0.653 0.518 0.499
    1             100     18    RF    0.658 0.524 0.501
    1             100     18    AB    0.576 0.285 0.826
    1             100     18   XGB    0.625 0.433 0.685
    2             100     18    LR    0.508 0.082 0.954
    2             100     18   KNN    0.738 0.645 0.107
    2             100     18    NB    0.626 0.55

In [ ]:
raw = pd.read_csv(P, usecols=["@timestamp", "target-class-name", "target-class"], low_memory=False)
print("raw rows:", len(raw))
bad = pd.to_datetime(raw["@timestamp"], errors="coerce").isna()
print("unparseable timestamps:", bad.sum())
print("sample bad values:", raw.loc[bad, "@timestamp"].head(10).tolist())
print("bad rows by class:", raw.loc[bad, "target-class"].value_counts().to_dict())
print("class share (all rows):", raw["target-class"].value_counts(normalize=True).round(3).to_dict())
print("class share (kept rows):", raw.loc[~bad, "target-class"].value_counts(normalize=True).round(3).to_dict())
print("files in folder:")
for f in sorted(glob.glob(f"{FOLDER}/**/*.csv", recursive=True)):
    print(" ", f.split("/")[-1])

raw rows: 350604
unparseable timestamps: 233161
sample bad values: ['24:00.4', '24:00.5', '24:00.5', '24:00.9', '24:00.9', '24:01.0', '24:01.1', '24:01.1', '24:01.1', '24:01.2']
bad rows by class: {0: 217780, 1: 15381}
class share (all rows): {0: 0.908, 1: 0.092}
class share (kept rows): {0: 0.855, 1: 0.145}
files in folder:
  dataset-imbalanced-10-all.csv
  dataset-imbalanced-10-engineered.csv
  dataset-imbalanced-10-fasttext.csv


In [ ]:
raw = pd.read_csv(P, usecols=["@timestamp"], low_memory=False)
s = raw["@timestamp"].astype(str)
print("pattern MM:SS.f count:", s.str.fullmatch(r"\d{2}:\d{2}\.\d+").sum())
print("pattern with date (ok):", s.str.contains(r"^\d{4}-").sum())
print("other:", (~s.str.fullmatch(r"\d{2}:\d{2}\.\d+") & ~s.str.contains(r"^\d{4}-")).sum())
print("max minutes in MM:SS:", s[s.str.fullmatch(r"\d{2}:\d{2}\.\d+")].str[:2].astype(int).max())

pattern MM:SS.f count: 350604
pattern with date (ok): 0
other: 0
max minutes in MM:SS: 59


In [ ]:
df = pd.read_csv(P, usecols=["@timestamp", "event.code", "target-class-name", "target-class"],
                 low_memory=False)
s = df["@timestamp"].astype(str)
mm = s.str[:2].astype(int)
ss = s.str[3:].astype(float)       # "SS.f" as seconds
df["t_sec"] = mm * 60 + ss          # seconds from the start of the file
print("rows:", len(df), "| time range (s):", round(df["t_sec"].min(), 1), "to", round(df["t_sec"].max(), 1))
print("class share:", df["target-class"].value_counts(normalize=True).round(3).to_dict())

rows: 350604 | time range (s): 0.0 to 3599.9
class share: {0: 0.908, 1: 0.092}


In [ ]:
diffs = df["t_sec"].diff().dropna()
print("negative steps (time going back):", int((diffs < 0).sum()))
print("largest jump (s):", round(diffs.max(), 2))

negative steps (time going back): 16
largest jump (s): 3126.6


In [ ]:
df = pd.read_csv(P, usecols=["@timestamp", "event.code", "target-class-name", "target-class"],
                 low_memory=False)

In [ ]:
s = df["@timestamp"].astype(str)
t_sec = s.str[:2].astype(int) * 60 + s.str[3:].astype(float)
block = (t_sec // 300).astype(int)   # ভাগের জন্য শুধু, ক্রমের জন্য নয়
df["grp"] = np.where(df["target-class-name"] == "goodware",
                     "goodware_b" + block.astype(str),
                     df["target-class-name"])
print("groups:", df["grp"].nunique())
print(df.groupby("grp")["target-class"].first().value_counts().to_dict())

groups: 32
{1: 20, 0: 12}


In [ ]:
X = pd.crosstab(df["grp"], df["event.code"])
X.columns = [f"ev_{int(c)}" for c in X.columns]
X = X.drop(columns=[c for c in ["ev_255"] if c in X.columns])
meta = df.groupby("grp").agg(y=("target-class", "first"))
meta = meta.loc[X.index]
X = X[X.columns[X.nunique() > 1]]

cols = list(X.columns)
Xv = X.values.astype(float)
yv = meta["y"].values.astype(int)
gv = X.index.values
print("feature table:", X.shape, "| labels (0,1):", np.bincount(yv))

feature table: (32, 18) | labels (0,1): [12 20]


In [ ]:
n_splits = min(5, len(np.unique(gv)))
gkf = GroupKFold(n_splits=n_splits)
folds = list(gkf.split(Xv, yv, gv))
for i, (tr, te) in enumerate(folds):
    print(f"fold {i}: train {len(tr)} (ransom {yv[tr].sum()}), "
          f"test {len(te)} (ransom {yv[te].sum()})")

fold 0: train 25 (ransom 16), test 7 (ransom 4)
fold 1: train 25 (ransom 16), test 7 (ransom 4)
fold 2: train 26 (ransom 16), test 6 (ransom 4)
fold 3: train 26 (ransom 16), test 6 (ransom 4)
fold 4: train 26 (ransom 16), test 6 (ransom 4)


In [ ]:
import pandas as pd
import numpy as np
import glob
import sys

FOLDER = "/content/drive/MyDrive/Thesis/radar/JamilIsp-RADAR-cb0c4c2/dataset-imbalanced-10"
csv_files = sorted(glob.glob(f"{FOLDER}/**/*.csv", recursive=True))

if not csv_files:
    print(f"Error: No CSV files found in {FOLDER}. Please verify your Google Drive is mounted and the folder path exists.")
else:
    P = csv_files[0]
    # Ensure df is loaded and processed
    df = pd.read_csv(P, usecols=["@timestamp", "event.code", "target-class-name", "target-class"], low_memory=False)
    s = df["@timestamp"].astype(str)
    t_sec = s.str[:2].astype(int) * 60 + s.str[3:].astype(float)
    block = (t_sec // 300).astype(int)
    df["grp"] = np.where(df["target-class-name"] == "goodware",
                         "goodware_b" + block.astype(str),
                         df["target-class-name"])

    X = pd.crosstab(df["grp"], df["event.code"])
    X.columns = [f"ev_{int(c)}" for c in X.columns]
    X = X.drop(columns=[c for c in ["ev_255"] if c in X.columns])
    X = X.div(X.sum(axis=1), axis=0)   # প্রতি run-এর event code-এর অনুপাত
    meta = df.groupby("grp").agg(y=("target-class", "first"))
    meta = meta.loc[X.index]
    X = X[X.columns[X.nunique() > 1]]

    cols = list(X.columns)
    Xv = X.values.astype(float)
    yv = meta["y"].values.astype(int)
    gv = X.index.values
    print("feature table:", X.shape, "| labels (0,1):", np.bincount(yv))

feature table: (32, 18) | labels (0,1): [12 20]


In [ ]:
n_splits = min(5, len(np.unique(gv)))
gkf = GroupKFold(n_splits=n_splits)
folds = list(gkf.split(Xv, yv, gv))
for i, (tr, te) in enumerate(folds):
    print(f"fold {i}: train {len(tr)} (ransom {yv[tr].sum()}), "
          f"test {len(te)} (ransom {yv[te].sum()})")

fold 0: train 25 (ransom 16), test 7 (ransom 4)
fold 1: train 25 (ransom 16), test 7 (ransom 4)
fold 2: train 26 (ransom 16), test 6 (ransom 4)
fold 3: train 26 (ransom 16), test 6 (ransom 4)
fold 4: train 26 (ransom 16), test 6 (ransom 4)


In [ ]:
p1_rows = []
for i, (tr, te) in enumerate(folds):
    sc = StandardScaler().fit(Xv[tr])
    _, sel, _ = paper1_votes(sc.transform(Xv[tr]), yv[tr])
    for name in make_models():
        bal, f1, fnr = fit_eval(Xv[tr], yv[tr], Xv[te], yv[te], sel, name)
        p1_rows.append({"fold": i, "n_sel": len(sel), "model": name,
                        "bal_acc": bal, "F1": f1, "FNR": fnr})
p1 = pd.DataFrame(p1_rows)
print(p1.groupby("model")[["bal_acc", "F1", "FNR"]].agg(["mean", "std"]).round(3))
print("selected per fold:", p1.drop_duplicates("fold")["n_sel"].tolist())

      bal_acc            F1          FNR       
         mean    std   mean    std  mean    std
model                                          
AB      0.967  0.075  0.978  0.050  0.00  0.000
DT      0.967  0.075  0.978  0.050  0.00  0.000
KNN     0.858  0.091  0.905  0.055  0.05  0.112
LR      0.908  0.085  0.927  0.068  0.05  0.112
NB      0.950  0.068  0.943  0.078  0.10  0.137
RF      0.917  0.118  0.956  0.061  0.00  0.000
XGB     0.967  0.075  0.978  0.050  0.00  0.000
selected per fold: [8, 8, 8, 9, 9]


In [ ]:
PCTS = list(range(10, 101, 10))

def n_for(p, total):
    return max(1, int(round(p / 100 * total)))

sweep_rows = []
for i, (tr, te) in enumerate(folds):
    sc = StandardScaler().fit(Xv[tr])
    order = order_from(sc.transform(Xv[tr]), yv[tr])
    for p in PCTS:
        idx = order[:n_for(p, len(order))]
        for name in make_models():
            bal, f1, fnr = fit_eval(Xv[tr], yv[tr], Xv[te], yv[te], idx, name)
            sweep_rows.append({"fold": i, "percent": p, "model": name,
                               "bal_acc": bal, "F1": f1, "FNR": fnr})
sweep = pd.DataFrame(sweep_rows)
print(sweep.groupby(["model", "percent"])["bal_acc"].agg(["mean", "std"])
            .round(3).unstack("model"))

          mean                                             std                \
model       AB     DT    KNN     LR    NB     RF    XGB     AB     DT    KNN   
percent                                                                        
10       0.967  0.967  0.733  0.758  0.95  0.967  0.967  0.075  0.075  0.253   
20       0.967  0.967  0.700  0.725  0.95  0.967  0.967  0.075  0.075  0.090   
30       0.967  0.967  0.700  0.808  0.95  0.917  0.967  0.075  0.075  0.090   
40       0.967  0.967  0.858  0.908  0.95  0.917  0.967  0.075  0.075  0.091   
50       0.967  0.967  0.858  0.908  0.95  0.917  0.967  0.075  0.075  0.091   
60       0.967  0.967  0.858  0.908  0.95  0.967  0.967  0.075  0.075  0.091   
70       0.967  0.967  0.933  0.908  0.95  0.967  0.967  0.075  0.075  0.091   
80       0.967  0.967  0.933  0.908  0.95  0.967  0.967  0.075  0.075  0.091   
90       0.967  0.967  0.933  0.908  0.95  0.917  0.967  0.075  0.075  0.091   
100      0.967  0.967  0.908  0.908  0.9

In [ ]:
def choose_percent(X_train, y_train, g_train, target_model_name="RF"):
    inner_folds = list(GroupKFold(n_splits=3).split(X_train, y_train, g_train))
    perf = {p: [] for p in PCTS}
    for itr, ite in inner_folds:
        sc = StandardScaler().fit(X_train[itr])
        order = order_from(sc.transform(X_train[itr]), y_train[itr])
        for p in PCTS:
            idx = order[:n_for(p, len(order))]
            model = make_models()[target_model_name]
            model.fit(X_train[itr][:, idx], y_train[itr])
            pred = model.predict(X_train[ite][:, idx])
            perf[p].append(balanced_accuracy_score(y_train[ite], pred))
    return max(PCTS, key=lambda p: np.mean(perf[p]))

final_rows = []
for i, (tr, te) in enumerate(folds):
    p_star = choose_percent(Xv[tr], yv[tr], gv[tr], "RF")
    sc = StandardScaler().fit(Xv[tr])
    order = order_from(sc.transform(Xv[tr]), yv[tr])
    idx = order[:n_for(p_star, len(order))]
    for name in make_models():
        bal, f1, fnr = fit_eval(Xv[tr], yv[tr], Xv[te], yv[te], idx, name)
        final_rows.append({"fold": i, "chosen_percent": p_star, "n_sel": len(idx),
                           "model": name, "bal_acc": bal, "F1": f1, "FNR": fnr})
final = pd.DataFrame(final_rows)
print(final.groupby("model")[["bal_acc", "F1", "FNR"]].agg(["mean", "std"]).round(3))
print("chosen percent per fold:", final.drop_duplicates("fold")["chosen_percent"].tolist())

      bal_acc            F1          FNR       
         mean    std   mean    std  mean    std
model                                          
AB      0.967  0.075  0.978  0.050  0.00  0.000
DT      0.967  0.075  0.978  0.050  0.00  0.000
KNN     0.883  0.162  0.938  0.091  0.00  0.000
LR      0.908  0.146  0.931  0.096  0.05  0.112
NB      0.950  0.068  0.943  0.078  0.10  0.137
RF      0.967  0.075  0.978  0.050  0.00  0.000
XGB     0.967  0.075  0.978  0.050  0.00  0.000
chosen percent per fold: [10, 10, 10, 50, 70]


In [ ]:
all_idx = np.arange(len(cols))
base_rows = []
for i, (tr, te) in enumerate(folds):
    for name in make_models():
        bal, f1, fnr = fit_eval(Xv[tr], yv[tr], Xv[te], yv[te], all_idx, name)
        base_rows.append({"fold": i, "model": name, "bal_acc": bal, "F1": f1, "FNR": fnr})
base = pd.DataFrame(base_rows)
print(base.groupby("model")[["bal_acc", "F1", "FNR"]].agg(["mean", "std"]).round(3))

      bal_acc            F1          FNR       
         mean    std   mean    std  mean    std
model                                          
AB      0.967  0.075  0.978  0.050  0.00  0.000
DT      0.967  0.075  0.978  0.050  0.00  0.000
KNN     0.908  0.085  0.927  0.068  0.05  0.112
LR      0.908  0.085  0.927  0.068  0.05  0.112
NB      0.950  0.068  0.943  0.078  0.10  0.137
RF      0.967  0.075  0.978  0.050  0.00  0.000
XGB     0.967  0.075  0.978  0.050  0.00  0.000


In [ ]:
summary = pd.DataFrame({
    "all_features": base.groupby("model")["bal_acc"].mean(),
    "paper1_rule": p1.groupby("model")["bal_acc"].mean(),
    "nested_sweep": final.groupby("model")["bal_acc"].mean(),
}).round(3)
print(summary)

       all_features  paper1_rule  nested_sweep
model                                         
AB            0.967        0.967         0.967
DT            0.967        0.967         0.967
KNN           0.908        0.858         0.883
LR            0.908        0.908         0.908
NB            0.950        0.950         0.950
RF            0.967        0.917         0.967
XGB           0.967        0.967         0.967


In [ ]:
sc_all = StandardScaler().fit(Xv)
votes_all, sel_all, mi_all = paper1_votes(sc_all.transform(Xv), yv)
order_all = np.lexsort((-mi_all, -votes_all))
sel_set = set(sel_all.tolist())
report = pd.DataFrame({
    "feature": [cols[j] for j in order_all],
    "votes": votes_all[order_all],
    "MI": mi_all[order_all].round(4),
    "in_paper_rule": [j in sel_set for j in order_all],
})
print(report.to_string(index=False))

sets = []
for tr, te in folds:
    sc = StandardScaler().fit(Xv[tr])
    _, sel, _ = paper1_votes(sc.transform(Xv[tr]), yv[tr])
    sets.append(set(sel.tolist()))
jac = [len(a & b) / len(a | b) for i, a in enumerate(sets) for b in sets[i + 1:]]
print("selected-set Jaccard, mean:", round(float(np.mean(jac)), 3))

feature  votes     MI  in_paper_rule
   ev_2      4 0.5882           True
  ev_11      4 0.4061           True
  ev_29      4 0.4019           True
   ev_4      4 0.2329           True
  ev_23      4 0.2193           True
   ev_6      4 0.2179           True
  ev_15      3 0.1230           True
  ev_13      2 0.0601           True
   ev_3      1 0.2788          False
  ev_10      1 0.2349          False
  ev_25      1 0.2089          False
  ev_12      1 0.0936          False
   ev_5      1 0.0618          False
   ev_8      1 0.0000          False
   ev_1      0 0.1655          False
  ev_22      0 0.0675          False
   ev_7      0 0.0357          False
  ev_17      0 0.0000          False
selected-set Jaccard, mean: 0.796


In [ ]:
print(final.round(3).to_string(index=False))

 fold  chosen_percent  n_sel model  bal_acc    F1  FNR
    0              10      2    LR    0.667 0.800 0.00
    0              10      2   KNN    0.667 0.800 0.00
    0              10      2    NB    1.000 1.000 0.00
    0              10      2    DT    1.000 1.000 0.00
    0              10      2    RF    1.000 1.000 0.00
    0              10      2    AB    1.000 1.000 0.00
    0              10      2   XGB    1.000 1.000 0.00
    1              10      2    LR    1.000 1.000 0.00
    1              10      2   KNN    1.000 1.000 0.00
    1              10      2    NB    0.875 0.857 0.25
    1              10      2    DT    0.833 0.889 0.00
    1              10      2    RF    0.833 0.889 0.00
    1              10      2    AB    0.833 0.889 0.00
    1              10      2   XGB    0.833 0.889 0.00
    2              10      2    LR    1.000 1.000 0.00
    2              10      2   KNN    1.000 1.000 0.00
    2              10      2    NB    1.000 1.000 0.00
    2     

In [ ]:
from sklearn.metrics import roc_auc_score
auc = pd.Series({c: roc_auc_score(yv, Xv[:, j]) for j, c in enumerate(cols)})
print(np.maximum(auc, 1 - auc).sort_values(ascending=False).round(3).to_string())

ev_2     1.000
ev_29    0.958
ev_11    0.908
ev_23    0.825
ev_3     0.821
ev_6     0.792
ev_4     0.792
ev_15    0.790
ev_5     0.746
ev_1     0.746
ev_13    0.712
ev_10    0.708
ev_25    0.673
ev_12    0.642
ev_17    0.600
ev_22    0.587
ev_8     0.583
ev_7     0.558


In [ ]:
tbl = X.copy()
tbl["n_events"] = df.groupby("grp").size().reindex(X.index)
tbl["y"] = meta["y"].values
print(tbl.round(3).sort_values("y").to_string())

                                                                              ev_1   ev_2   ev_3  ev_4   ev_5   ev_6   ev_7  ev_8  ev_10  ev_11  ev_12  ev_13  ev_15  ev_17  ev_22  ev_23  ev_25  ev_29  n_events  y
grp                                                                                                                                                                                                                 
goodware_b2                                                                  0.020  0.238  0.007   0.0  0.009  0.000  0.076   0.0  0.032  0.353  0.023  0.149  0.001  0.032  0.014  0.025  0.000  0.020     12763  0
goodware_b3                                                                  0.018  0.335  0.004   0.0  0.004  0.001  0.038   0.0  0.026  0.391  0.018  0.110  0.000  0.017  0.005  0.030  0.000  0.005     20993  0
goodware_b4                                                                  0.012  0.312  0.005   0.0  0.004  0.000  0.042   0.0  0.028  0.468  0.0

In [ ]:
print(X.assign(y=meta["y"].values)
        .groupby("y")[["ev_2", "ev_11", "ev_29", "ev_23"]]
        .agg(["min", "max", "mean"]).round(3).T)

y               0      1
ev_2  min   0.002  0.000
      max   0.374  0.001
      mean  0.179  0.000
ev_11 min   0.315  0.000
      max   0.861  0.832
      mean  0.486  0.236
ev_29 min   0.000  0.000
      max   0.048  0.001
      mean  0.010  0.000
ev_23 min   0.015  0.000
      max   0.343  0.915
      mean  0.080  0.310
